### 始めはpythonを用いて共起ネットワークを作成したかったが、なかなかうまくいかなかったためKHCoderを用いて作成する 

In [16]:
from pathlib import Path
import collections
import MeCab
import neologdn
import unicodedata
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations, dropwhile
from collections import Counter, OrderedDict
import numpy as np
from networkx.drawing import nx_agraph

In [17]:
import pandas as pd
df = pd.read_csv('news.csv')
lines = df.loc[40:49, '本文']
stopword_list = ['|','で','い','た','ある','よう','ない', 'かた', 'ため', 'き','それ','なく','じゃ','わい','う','の','だ','な','れ','ず','さっき','これ','事','一','人']

# "stopword_list="によって、インデックスしない単語を指定


In [3]:
# 分かち書きを行う（辞書ディレクトリはご自身のディレクトリを指定ください）

mecabTagger = MeCab.Tagger("-Owakati") 
select_conditions = ['動詞', '形容詞', '名詞','副詞', '助動詞','感動詞']
noun_sentences = []
for sentence in lines:
    words = []
    sentence = neologdn.normalize(sentence)
    sentence = unicodedata.normalize("NFKC", sentence)
    node = mecabTagger.parseToNode(sentence).next
    while node:
        word = node.surface
        pos1 = node.feature.split(',')[0]
        pos2 = node.feature.split(',')[1]
        pos3 = node.feature.split(',')[2]
        if pos1 in select_conditions and word not in stopword_list:
            if not (pos1 in "動詞" and pos2 in "非自立"):                 
                words.append(node.surface) # 単語
        node = node.next
    noun_sentences.append(words)

In [4]:
# jaccard係数を計算する（jaccard係数：0.12以上、章跨ぎ単語登場数：4以上）
jaccard_coef = []
edge_th=0.12
pair_all = []
min_cnt=4
print('共起ネットワーク用単語ペア')
for chapter in noun_sentences:
    pair_temp = list(combinations(set(chapter), 2))
    for i,pair in enumerate(pair_temp):
        pair_temp[i] = tuple(sorted(pair))
    pair_all += pair_temp
pair_count = Counter(pair_all)
for key, count in dropwhile(lambda key_count: key_count[1] >= min_cnt, pair_count.most_common()):
    del pair_count[key]
word_count = Counter()
for chapter in noun_sentences:
    word_count += Counter(set(chapter))
for pair, cnt in pair_count.items():
    jaccard_coef.append(cnt / (word_count[pair[0]] + word_count[pair[1]] - cnt))
print('単語ペア', '出現数', 'jaccard係数', '単語１出現数', '単語２出現数', sep='\t')
jaccard_dict = OrderedDict()
for (pair, cnt), coef in zip(pair_count.items(), jaccard_coef):
    if coef >= edge_th:
        jaccard_dict[pair] = coef
        print(pair, cnt, coef, word_count[pair[0]], word_count[pair[1]], sep='\t')

共起ネットワーク用単語ペア
単語ペア	出現数	jaccard係数	単語１出現数	単語２出現数
('10', 'ます')	4	0.4	4	10
('10', 'です')	4	0.4444444444444444	4	9
('10', 'まし')	4	0.4444444444444444	4	9
('さ', 'ます')	7	0.7	7	10
('なっ', 'ます')	5	0.5	5	10
('3', 'ます')	4	0.4	4	10
('ます', '発表')	5	0.5	10	5
('です', 'ます')	9	0.9	9	10
('まし', 'ます')	9	0.9	9	10
('ます', '行わ')	4	0.4	10	4
('2', 'ます')	5	0.5	5	10
('さ', 'なっ')	5	0.7142857142857143	7	5
('さ', '発表')	5	0.7142857142857143	7	5
('さ', 'です')	6	0.6	7	9
('さ', 'まし')	7	0.7777777777777778	7	9
('なっ', '発表')	4	0.6666666666666666	5	5
('です', 'なっ')	4	0.4	9	5
('なっ', 'まし')	5	0.5555555555555556	5	9
('3', 'まし')	4	0.4444444444444444	4	9
('2', '3')	4	0.8	5	4
('です', '発表')	4	0.4	9	5
('まし', '発表')	5	0.5555555555555556	9	5
('です', 'まし')	8	0.8	9	9
('2', 'です')	4	0.4	5	9
('2', 'まし')	5	0.5555555555555556	5	9
('ます', '指摘')	4	0.4	10	4
('いう', 'ます')	6	0.6	6	10
('なる', 'ます')	4	0.4	4	10
('し', 'ます')	9	0.9	9	10
('こと', 'ます')	7	0.7	7	10
('いる', 'ます')	7	0.7	7	10
('ます', '年')	6	0.6	10	6
('する', 'ます')	4	0.4	4	10
('ます', '施設')	4	0.4	10	4
('まし', '指摘')	4	0.

In [5]:
# 共起ネットワークを作成する
G = nx.Graph()
nodes = sorted(set([j for pair in jaccard_dict.keys() for j in pair]))
G.add_nodes_from(nodes)
print('Number of nodes=', G.number_of_nodes())
for pair, coef in jaccard_dict.items():
    G.add_edge(pair[0], pair[1], weight=coef)
print('Number of edges=', G.number_of_edges())
plt.figure(figsize=(15, 15))
seed = 0
np.random.seed(seed)
pos = nx_agraph.graphviz_layout(G, prog='neato', args='-Goverlap="scalexy" -Gsep="+6" -Gnodesep=0.8 -Gsplines="polyline" -GpackMode="graph" -Gstart={}'.format(seed))
pr = nx.pagerank(G)
nx.draw_networkx_nodes(G, pos, node_color=list(pr.values()), cmap=plt.cm.rainbow, alpha=0.7, node_size=[100000*v for v in pr.values()])
nx.draw_networkx_labels(G, pos, font_family='MS Gothic', font_weight='bold')
edge_width = [d['weight'] * 8 for (u, v, d) in G.edges(data=True)]
nx.draw_networkx_edges(G, pos, alpha=0.7, edge_color='darkgrey', width=edge_width)
plt.axis('off')
plt.tight_layout()
plt.savefig('co-occurance.png', bbox_inches='tight')

Number of nodes= 20
Number of edges= 86


ImportError: requires pygraphviz http://pygraphviz.github.io/

<Figure size 1500x1500 with 0 Axes>

In [2]:
pip show graphviz


Name: graphviz
Version: 0.20.3
Summary: Simple Python interface for Graphviz
Home-page: https://github.com/xflr6/graphviz
Author: Sebastian Bank
Author-email: sebastian.bank@uni-leipzig.de
License: MIT
Location: C:\Users\22t312\AppData\Local\Programs\Python\Python311\Lib\site-packages
Requires: 
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
df = pd.read_csv('ニュースサイト関連/news.csv')

In [ ]:
df_01_20 = df[df['日付'] == '2025-01-20']
df_01_20.to_csv('KHCorder_csvファイル/news_01_20.csv', index=False, encoding='utf-8-sig')

In [ ]:
df_sentence = df_01_20['本文']
df_sentence.to_csv('KHCorder_csvファイル/01_20_sentence.csv', index=False, encoding='utf-8-sig')